# Udaplay - Part 1: Vector Database & Data Ingestion

This notebook loads the local dataset of video game information (provided as individual JSON files),
cleans and formats it into RAG-ready documents, embeds those documents, and stores them in a
**persistent ChromaDB vector database** so they can later be retrieved with semantic search.

**Steps covered:**
1. Load the raw per-game JSON files from `games/`
2. Clean and format each record into a single natural-language document + metadata
3. Create a persistent Chroma collection with a sentence-embedding function
4. Add all documents (with embeddings + metadata) to the collection
5. Query the collection with natural-language questions to demonstrate semantic search


In [17]:
import json
import os
import glob

import chromadb
from chromadb.utils import embedding_functions


## 1. Load the raw dataset

Each game is provided as its own JSON file under `games/`.

In [18]:
import glob
import os
import json

GAMES_DIR = "games"

raw_games = []
for filepath in sorted(glob.glob(os.path.join(GAMES_DIR, "*.json"))):
    with open(filepath, "r", encoding="utf-8") as f:
        game = json.load(f)
    if isinstance(game, list):
        # Handle files that contain a list of game records
        for item in game:
            if isinstance(item, dict):
                item["_source_file"] = os.path.basename(filepath)
                raw_games.append(item)
        continue

    if isinstance(game, dict):
        # Handle files that contain a single game record
        game["_source_file"] = os.path.basename(filepath)
    else:
        # Skip unexpected JSON shapes
        continue
    raw_games.append(game)

print(f"Loaded {len(raw_games)} game records")
raw_games[0]


Loaded 20 game records


{'Name': 'Gran Turismo',
 'Platform': 'PlayStation 1',
 'Genre': 'Racing',
 'Publisher': 'Sony Computer Entertainment',
 'YearOfRelease': 1997,
 'Description': 'A realistic racing simulator featuring a large roster of licensed cars, detailed physics, and a career mode that lets players earn licenses and upgrade vehicles.',
 '_source_file': 'games_dataset.json'}

## 2. Clean and format the data

For each game we:
- normalize/validate the expected fields (`Name`, `Platform`, `Genre`, `Publisher`, `YearOfRelease`, `Description`)
- build a single natural-language **document string** that reads well for embedding and retrieval
- keep the structured fields as **metadata** so we can filter/inspect results later


In [ ]:
def clean_text(value):
    """Basic text cleanup: strip whitespace, collapse internal spaces."""
    if value is None:
        return ""
    return " ".join(str(value).split())


def format_game_document(game):
    """Turn a structured game record into one RAG-friendly text chunk."""
    name = clean_text(game.get("Name"))
    platform = clean_text(game.get("Platform"))
    genre = clean_text(game.get("Genre"))
    publisher = clean_text(game.get("Publisher"))
    year = clean_text(game.get("YearOfRelease"))
    description = clean_text(game.get("Description"))

    return (
        f"{name} is a {genre} game released in {year} for {platform}, "
        f"published by {publisher}. {description}"
    )


documents = []      # the text chunks that get embedded
metadatas = []       # structured metadata per chunk
ids = []             # unique id per chunk

for i, game in enumerate(raw_games):
    doc_text = format_game_document(game)
    documents.append(doc_text)
    metadatas.append({
        "Name": clean_text(game.get("Name")),
        "Platform": clean_text(game.get("Platform")),
        "Genre": clean_text(game.get("Genre")),
        "Publisher": clean_text(game.get("Publisher")),
        "YearOfRelease": game.get("YearOfRelease"),
        "source_file": game.get("_source_file"),
    })
    ids.append(f"game_{i:03d}")

print(documents[0])


Portal 2 is a Puzzle game released in 2011 for PC, published by Valve Corporation. Chell escapes Aperture Science using a portal gun to solve physics-based puzzles, joined by the sardonic AI GLaDOS and co-op robots Atlas and P-Body.


## 3. Create a persistent Chroma vector database

We use `chromadb.PersistentClient` so the collection is written to disk (`./chroma_db`) and
survives across notebook restarts, rather than living only in memory.

We use OpenAI's `text-embedding-3-small` model (via `OpenAIEmbeddingFunction`) to turn each
document into a dense embedding vector. This only needs a lightweight HTTPS call per batch of
text (no multi-hundred-MB model download), so it works well even in restricted network
environments.

**API key handling:** the key is never hard-coded in this notebook. It's read from the
`OPENAI_API_KEY` environment variable if set, otherwise you'll be prompted to enter it securely
(hidden input, not echoed or stored in the notebook file).


In [20]:
import os
import getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]


In [21]:
import chromadb
from chromadb.utils import embedding_functions

CHROMA_PATH = "chroma_db"
COLLECTION_NAME = "video_games"

client = chromadb.PersistentClient(path=CHROMA_PATH)

embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=OPENAI_API_KEY,
    model_name="text-embedding-3-small",
)

# get_or_create so re-running the notebook doesn't error on a duplicate collection
collection = client.get_or_create_collection(
    name=COLLECTION_NAME,
    embedding_function=embedding_fn,
    metadata={"hnsw:space": "cosine"},
)

print("Collection ready:", collection.name)


Collection ready: video_games


## 4. Add the processed documents to the vector database

In [22]:
# Upsert so this cell is safe to re-run without creating duplicate entries
collection.upsert(
    ids=ids,
    documents=documents,
    metadatas=metadatas,
)

print(f"Collection now contains {collection.count()} documents")


Collection now contains 20 documents


## 5. Query the vector database with semantic search

These queries use natural language rather than exact keywords, showing that retrieval is based on
**meaning**, not just string matching.


In [23]:
def semantic_search(query, n_results=3):
    results = collection.query(query_texts=[query], n_results=n_results)
    print(f"Query: {query!r}\n")
    for rank, (doc, meta, dist) in enumerate(
        zip(results["documents"][0], results["metadatas"][0], results["distances"][0]), start=1
    ):
        print(f"{rank}. {meta['Name']} ({meta['Platform']}, {meta['YearOfRelease']}) "
              f"- distance={dist:.4f}")
        print(f"   {doc}\n")


semantic_search("an open world cowboy game")


Query: 'an open world cowboy game'

1. Red Dead Redemption 2 (Xbox One, 2018) - distance=0.5603
   Red Dead Redemption 2 is a Action-Adventure game released in 2018 for Xbox One, published by Rockstar Games. Outlaw Arthur Morgan navigates the decline of the Wild West as his gang faces internal betrayal and encroaching civilization in a richly detailed frontier.

2. Elden Ring (PlayStation 5, 2022) - distance=0.6536
   Elden Ring is a Action RPG game released in 2022 for PlayStation 5, published by Bandai Namco Entertainment. An open-world dark fantasy epic co-created with George R. R. Martin, challenging players with punishing combat and a mysterious, sprawling Lands Between.

3. The Witcher 3: Wild Hunt (PC, 2015) - distance=0.6538
   The Witcher 3: Wild Hunt is a Action RPG game released in 2015 for PC, published by CD Projekt. Monster hunter Geralt of Rivia searches for his adopted daughter across a vast open world shaped by morally complex choices and rich side quests.



In [24]:
semantic_search("racing games for PlayStation")


Query: 'racing games for PlayStation'

1. Gran Turismo (PlayStation 1, 1997) - distance=0.4072
   Gran Turismo is a Racing game released in 1997 for PlayStation 1, published by Sony Computer Entertainment. A realistic racing simulator featuring a large roster of licensed cars, detailed physics, and a career mode that lets players earn licenses and upgrade vehicles.

2. Grand Theft Auto V (PlayStation 3, 2013) - distance=0.6403
   Grand Theft Auto V is a Action-Adventure game released in 2013 for PlayStation 3, published by Rockstar Games. An open-world crime saga set in Los Santos following three protagonists whose stories intertwine through heists, satire, and sprawling exploration.

3. Mario Kart 8 Deluxe (Nintendo Switch, 2017) - distance=0.6471
   Mario Kart 8 Deluxe is a Racing game released in 2017 for Nintendo Switch, published by Nintendo. An enhanced kart racer with anti-gravity tracks, a large roster of Nintendo characters, and robust local and online multiplayer modes.



In [25]:
semantic_search("a Nintendo platformer with Mario")


Query: 'a Nintendo platformer with Mario'

1. Super Mario World (Super Nintendo Entertainment System, 1990) - distance=0.4101
   Super Mario World is a Platformer game released in 1990 for Super Nintendo Entertainment System, published by Nintendo. A side-scrolling platformer starring Mario and Yoshi as they traverse Dinosaur Land to rescue Princess Toadstool from Bowser.

2. Mario Kart 8 Deluxe (Nintendo Switch, 2017) - distance=0.5568
   Mario Kart 8 Deluxe is a Racing game released in 2017 for Nintendo Switch, published by Nintendo. An enhanced kart racer with anti-gravity tracks, a large roster of Nintendo characters, and robust local and online multiplayer modes.

3. The Legend of Zelda: Ocarina of Time (Nintendo 64, 1998) - distance=0.6420
   The Legend of Zelda: Ocarina of Time is a Action-Adventure game released in 1998 for Nintendo 64, published by Nintendo. Link travels between childhood and adulthood in Hyrule, wielding the Ocarina of Time to solve puzzles and defeat Ganondo

In [26]:
semantic_search("post-apocalyptic survival story with strong characters")


Query: 'post-apocalyptic survival story with strong characters'

1. The Last of Us (PlayStation 3, 2013) - distance=0.7127
   The Last of Us is a Action-Adventure game released in 2013 for PlayStation 3, published by Sony Computer Entertainment. Joel and Ellie navigate a fungal-infection apocalypse across the ruins of the United States, blending stealth combat with an emotionally driven narrative.

2. Red Dead Redemption 2 (Xbox One, 2018) - distance=0.7392
   Red Dead Redemption 2 is a Action-Adventure game released in 2018 for Xbox One, published by Rockstar Games. Outlaw Arthur Morgan navigates the decline of the Wild West as his gang faces internal betrayal and encroaching civilization in a richly detailed frontier.

3. Doom (2016) (PC, 2016) - distance=0.7490
   Doom (2016) is a First-Person Shooter game released in 2016 for PC, published by Bethesda Softworks. The Doom Slayer tears through demonic hordes on Mars in a fast-paced revival emphasizing aggressive movement and glory ki

## Summary

- Loaded the local video game dataset from individual JSON files
- Cleaned and formatted each record into a RAG-ready document + metadata
- Embedded the documents with OpenAI's `text-embedding-3-small` model and stored them in a
  **persistent** ChromaDB collection (`./chroma_db`)
- Demonstrated that the collection supports **semantic search**: queries return relevant games even
  when the wording doesn't exactly match the stored text

This vector database is now ready to be used as the retrieval component of a RAG pipeline
(Part 2: building the agent that queries it and generates answers).